# EFRiskEngine — AI Analysis of Unstructured Behavioural Data

This notebook deep-dives into the **AI-assisted** component of EFRiskEngine:
- Prompt engineering for fraud analysis
- Gemini Inference API integration
- Emerging pattern detection
- Score blending (rules + AI)

---
> Set `LLM_PROVIDER=gemini` in your `.env` file to make live API calls.

In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv(os.path.join(ROOT, '.env'))

import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
print('✅ Setup OK. Provider:', os.environ.get('LLM_PROVIDER', 'gemini'))

## 1. Load Scored Data

In [ ]:
from src.data_pipeline import run_etl
from src.risk_engine import run_risk_engine

users_df, txns_df = run_etl()
scored_txns, user_summaries = run_risk_engine(txns_df, users_df)

high_risk = user_summaries[user_summaries['user_risk_score'] >= 50]
print(f'High-risk users to analyse: {len(high_risk)}')

## 2. Initialise LLM Provider

In [ ]:
from src.ai_assist import get_provider, GeminiProvider, MockProvider

# Auto-select provider based on .env  (falls back to mock if token missing)
try:
    provider = get_provider()
    # Test with a quick ping
    test_resp = provider.complete('Say: ONLINE', max_tokens=10)
    print(f'Provider: {provider.name}')
    print(f'Test response: {test_resp[:80]}')
except Exception as e:
    print(f'Live provider unavailable ({e}), switching to mock')
    provider = MockProvider()

## 3. Single User Deep-Dive Prompt

In [ ]:
from src.ai_assist import analyze_user_behavior, _build_user_behavior_prompt

# Pick the single highest-risk user
top_user_id = user_summaries.iloc[0]['user_id']
user_row = users_df[users_df['user_id'] == top_user_id].iloc[0]
user_txns = scored_txns[scored_txns['user_id'] == top_user_id]
rules_score = int(user_summaries.iloc[0]['user_risk_score'])

print(f'Analysing user: {top_user_id} (rules score={rules_score})')
print('\n--- PROMPT SENT TO LLM ---')
print(_build_user_behavior_prompt(user_row, user_txns, rules_score))

In [ ]:
result = analyze_user_behavior(user_row, user_txns, provider, rules_score)
display(Markdown(f'### AI Analysis for `{top_user_id}`'))
display(Markdown(result['ai_analysis']))

## 4. Emerging Pattern Detection

In [ ]:
from src.ai_assist import detect_emerging_patterns, _build_pattern_detection_prompt

flagged_high = scored_txns[scored_txns['risk_score'] >= 50]
print(f'Flagged transactions sent to pattern detector: {len(flagged_high)}')
print('\n--- PROMPT ---')
print(_build_pattern_detection_prompt(flagged_high))

In [ ]:
patterns = detect_emerging_patterns(flagged_high, provider)
display(Markdown('### Emerging Fraud Patterns Identified'))
display(Markdown(patterns))

## 5. Score Blending — Rules + AI

In [ ]:
from src.ai_assist import combine_with_rules, run_ai_analysis

# Run on top 5 users and show blended scores
blended = run_ai_analysis(scored_txns, users_df, provider, top_n=5)
display(blended[['user_id','rules_risk_score','ai_bias_score','combined_score','combined_label']])

# Visualise the score comparison
fig = px.scatter(
    blended,
    x='rules_risk_score', y='combined_score',
    text='user_id', color='combined_label',
    color_discrete_map={'Low':'#22c55e','Medium':'#eab308','High':'#f97316','Critical':'#ef4444'},
    title='Rules Score vs Combined (Rules + AI) Score',
    labels={'rules_risk_score': 'Rules-Only Score', 'combined_score': 'Combined Score'}
)
fig.add_shape(type='line', x0=0, y0=0, x1=100, y1=100,
              line=dict(color='gray', dash='dash'))
fig.show()

## 6. Summary

The AI module adds qualitative context to the purely numerical rules score:
- For users where the rules score is already high, the AI tends to corroborate with **HIGH** or **CRITICAL**.
- For borderline (Medium) users the AI can push the combined score either direction based on narrative signals.
- The blending weight (75% rules, 25% AI) is tunable in `combine_with_rules(ai_weight=0.25)`.

### Switching Provider
```python
# .env
LLM_PROVIDER=openai        # or anthropic, mock
OPENAI_API_KEY=sk-...
```